In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

  Using cached qiskit-1.2.4-cp38-abi3-win_amd64.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached symengine-0.13.0.tar.gz (114 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-win_amd64.whl (4.6 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Failed to build symengine
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [19 lines of output]
  C:\Users\travi\AppData\Local\Temp\pip-build-env-6nxgox8a\overlay\Lib\site-packages\setuptools\_vendor\wheel\bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
    warn(
  C:\Users\travi\AppData\Local\Temp\pip-build-env-6nxgox8a\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
  !!
  
          ********************************************************************************
          Please consider removing the following classifiers in favor of a SPDX license expression:
  
          License :: OSI Approved :: MIT License
  
          See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
       

  Using cached qiskit-aer-0.15.1.tar.gz (6.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build qiskit-aer
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [410 lines of output]
  C:\Users\travi\AppData\Local\Temp\pip-build-env-l9ebqdde\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
  !!
  
          ********************************************************************************
          Please consider removing the following classifiers in favor of a SPDX license expression:
  
          License :: OSI Approved :: Apache Software License
  
          See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
          ********************************************************************************
  
  !!
    self._finalize_license_expression()
  
  
  --------------------------------------------------------------------------------
  -- Trying 'Ninja (Visual Studio 17 2022 x64 v144)' generator
  --------------------------------
  ---------------------------
  -------------

Note: you may need to restart the kernel to use updated packages.


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol WITH an attacker (Eve).

In [13]:
def generate_random_bits(n):
    """
    Generate n random bits using quantum measurements.
    Each bit is generated by:
    1. Creating a qubit in superposition using Hadamard gate
    2. Measuring the qubit to get a random 0 or 1
    """
    qc = QuantumCircuit(1, 1)
    
    # Put qubit in superposition: |0⟩ → (|0⟩ + |1⟩)/√2
    qc.h(0)
    
    # Measure the qubit
    qc.measure(0, 0)
    
    # Use BasicSimulator to execute the circuit n times
    simulator = BasicSimulator()
    
    # Execute circuit n times to get n random bits
    job = simulator.run(qc, shots=n)
    result = job.result()
    counts = result.get_counts()
    
    # Extract bits from measurement results
    random_bits = []
    for bit_str, count in counts.items():
        random_bits.extend([int(bit_str)] * count)
    
    # Shuffle to ensure randomness and trim to exactly n bits
    import random
    random.shuffle(random_bits)
    return random_bits[:n]

# Example: Generate random bits for Alice and Bob
n_bits = 256

# ========== ALICE'S SETUP ==========   
# Alice's random bit choices (0 or 1) - the data she wants to transmit
alice_bits = generate_random_bits(n_bits)
print(f"Alice's random bits: {alice_bits}")

# Alice's random basis choices (0 = rectilinear, 1 = diagonal) - how she encodes each bit
alice_bases = generate_random_bits(n_bits)
print(f"Alice's random bases: {alice_bases}")

# ========== BOB'S SETUP ==========
# Bob's random basis choices (0 = rectilinear, 1 = diagonal) - how he measures each qubit
bob_bases = generate_random_bits(n_bits)
print(f"Bob's random bases: {bob_bases}")

Alice's random bits: [0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0]
Alice's random bases: [1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0

In [14]:
# ========== STEP 1B: EVE'S ATTACK (EAVESDROPPING) ==========
# Eve intercepts Alice's qubits and measures them with random bases
# She then re-transmits the collapsed qubits to Bob
# This introduces errors that can be detected later

print("\n" + "="*60)
print("EVE'S EAVESDROPPING ATTACK")
print("="*60)

# ========== EVE'S SETUP ==========
# Eve generates random basis choices for her interception attempt
eve_bases = generate_random_bits(n_bits)
print(f"\nEve's random bases: {eve_bases}")

# ========== EVE MEASURES THE QUBITS ==========
# Eve measures each qubit with her randomly chosen bases
# If she guesses the correct basis, she gets the correct bit
# If she guesses wrong, she gets a random result and collapses the qubit incorrectly
eve_measured_bits = []

for i in range(n_bits):
    if eve_bases[i] == alice_bases[i]:
        # Eve guessed the correct basis - she gets the right bit
        eve_measured_bits.append(alice_bits[i])
    else:
        # Eve guessed wrong - she gets a random bit (50/50 chance)
        # This also collapses the qubit incorrectly!
        eve_measured_bits.append(generate_random_bits(1)[0])

print(f"\nEve's measured bits: {eve_measured_bits}")

# ========== EVE RE-TRANSMITS TO BOB ==========
# Eve sends the collapsed qubits to Bob
# But now some bits are corrupted due to her wrong basis measurements
print("\n[EVE sends intercepted (and partially corrupted) qubits to Bob]")

# ========== TRACK EVE'S ERRORS ==========
# Count where Eve's measurements differed from Alice's original bits
eve_errors = []
for i in range(n_bits):
    if eve_measured_bits[i] != alice_bits[i]:
        eve_errors.append(i)

print(f"\nPositions where Eve's measurements corrupted bits: {len(eve_errors)}/{n_bits}")
print(f"Error rate introduced by Eve: {100 * len(eve_errors) / n_bits:.1f}%")


EVE'S EAVESDROPPING ATTACK

Eve's random bases: [0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0]

Eve's measured bits: [1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 

In [15]:
# ========== STEP 2: ALICE AND BOB COMPARE BASES (WITH EVE'S INTERFERENCE) ==========
# After Alice transmits and Bob measures, they publicly compare their basis choices
# Bob received CORRUPTED qubits from Eve, not Alice's original qubits
# Only bits where Bob used the SAME basis as Alice will be correct (sift key)
# But some may be wrong due to Eve's eavesdropping!

matching_indices = []
sift_key = []
eve_introduced_errors = []

# Compare Alice's encoding bases with Bob's measurement bases
# Note: Bob is measuring Eve's re-transmitted qubits (which may be corrupted)
for i in range(n_bits):
    if alice_bases[i] == bob_bases[i]:  # Bases match
        matching_indices.append(i)
        # Bob received eve_measured_bits (Eve's measurements), not alice_bits
        sift_key.append(eve_measured_bits[i])
        
        # Check if Eve's eavesdropping introduced an error
        if eve_measured_bits[i] != alice_bits[i]:
            eve_introduced_errors.append(i)

print("\n=== BASIS COMPARISON (WITH EAVESDROPPER EVE) ===")
print(f"Position | Alice Base | Bob Base | Match? | Eve Error?")
print("-" * 60)
for i in range(n_bits):
    match = "✓ YES" if alice_bases[i] == bob_bases[i] else "✗ NO"
    eve_error = "⚠ EVE!" if i in eve_introduced_errors else "OK"
    basis_names = {0: "Rectilinear (+)", 1: "Diagonal (×)"}
    print(f"   {i}    | {basis_names[alice_bases[i]]:14} | {basis_names[bob_bases[i]]:14} | {match:6} | {eve_error}")

print(f"\n=== SIFT KEY RESULTS (WITH EVE'S INTERFERENCE) ===")
print(f"Positions with matching bases: {matching_indices}")
print(f"Number of matches: {len(matching_indices)} out of {n_bits}")
print(f"Errors Eve introduced in sift key: {len(eve_introduced_errors)}")
print(f"Error rate in sift key: {100 * len(eve_introduced_errors) / max(1, len(matching_indices)):.1f}%")
print(f"Sift key (corrupted by Eve): {sift_key}")
print(f"Sift key length: {len(sift_key)} bits")


=== BASIS COMPARISON (WITH EAVESDROPPER EVE) ===
Position | Alice Base | Bob Base | Match? | Eve Error?
------------------------------------------------------------
   0    | Diagonal (×)   | Rectilinear (+) | ✗ NO   | OK
   1    | Rectilinear (+) | Rectilinear (+) | ✓ YES  | ⚠ EVE!
   2    | Diagonal (×)   | Diagonal (×)   | ✓ YES  | ⚠ EVE!
   3    | Rectilinear (+) | Diagonal (×)   | ✗ NO   | OK
   4    | Rectilinear (+) | Rectilinear (+) | ✓ YES  | OK
   5    | Diagonal (×)   | Rectilinear (+) | ✗ NO   | OK
   6    | Rectilinear (+) | Diagonal (×)   | ✗ NO   | OK
   7    | Rectilinear (+) | Diagonal (×)   | ✗ NO   | OK
   8    | Diagonal (×)   | Rectilinear (+) | ✗ NO   | OK
   9    | Rectilinear (+) | Rectilinear (+) | ✓ YES  | OK
   10    | Diagonal (×)   | Rectilinear (+) | ✗ NO   | OK
   11    | Rectilinear (+) | Diagonal (×)   | ✗ NO   | OK
   12    | Rectilinear (+) | Diagonal (×)   | ✗ NO   | OK
   13    | Rectilinear (+) | Rectilinear (+) | ✓ YES  | OK
   14    | Diagonal (

In [20]:
# ========== STEP 3: EAVESDROPPING DETECTION (EVE GETS CAUGHT!) ==========
# Alice and Bob sacrifice some of their sift key bits to publicly verify there was no eavesdropping
# Eve's random basis guessing will cause detectable errors (~25% error rate)

import random

if len(sift_key) > 0:
    # ========== ALICE'S ACTION ==========
    # Alice selects random positions from the sift key to test
    num_test_bits = max(1, len(sift_key) // 3)  # Use ~1/3 of sift key for eavesdropping detection
    test_positions = sorted(random.sample(range(len(sift_key)), num_test_bits), reverse=True)
    
    # Map test positions (indices into sift_key) back to original qubit indices via matching_indices
    positions_in_matching = list(test_positions)
    original_indices = [matching_indices[pos] for pos in positions_in_matching]
    
    # Alice's test bits (from original alice_bits at those indices)
    test_bits_alice = [alice_bits[idx] for idx in original_indices]
    
    # Bob's test bits are what Bob actually received: eve_measured_bits at those original indices
    test_bits_bob = [eve_measured_bits[idx] for idx in original_indices]
    
    # Remove the tested bits from the sift_key (pop by positions in descending order)
    for pos in sorted(positions_in_matching, reverse=True):
        sift_key.pop(pos)
    
    # ========== ALICE AND BOB COMPARE ==========
    # Both parties publicly compare their test bits to check for mismatches
    # If Eve eavesdropped, we expect ~25% errors
    mismatches = sum(1 for a, b in zip(test_bits_alice, test_bits_bob) if a != b)
    error_rate = 100 * mismatches / max(1, len(test_bits_alice))
    
    print("\n=== EAVESDROPPING DETECTION (REVEALING EVE'S ATTACK) ===")
    print(f"Test positions selected (in sift_key): {sorted(test_positions, reverse=True)}")
    print(f"Original qubit indices tested: {original_indices}")
    print(f"\nAlice's original bits: {test_bits_alice}")
    print(f"Bob's received bits:   {test_bits_bob}")
    print(f"Mismatches detected: {mismatches} out of {num_test_bits} ({error_rate:.1f}%)")
    
    print(f"\n=== EAVESDROPPER DETECTION RESULT ===")
    # Threshold for eavesdropping detection (typically ~10-15%)
    EAVESDROP_THRESHOLD = 10.0
    
    if error_rate > EAVESDROP_THRESHOLD:
        print(f"⚠️  EVE DETECTED! ⚠️")
        print(f"Error rate {error_rate:.1f}% exceeds threshold {EAVESDROP_THRESHOLD}%")
        print(f"Expected error rate with eavesdropper: ~25% (basis guessing)")
        print(f"ACTION: ABORT PROTOCOL - KEY EXCHANGE FAILED")
        protocol_success = False
    else:
        print(f"✓ No eavesdropper detected (error rate {error_rate:.1f}% < {EAVESDROP_THRESHOLD}%)")
        print(f"ACTION: Continue - establish final key")
        protocol_success = True
    
    print(f"\n=== FINAL SECURE KEY STATUS ===")
    print(f"Remaining bits (after removing test bits): {sift_key}")
    print(f"Final key length: {len(sift_key)} bits")
    
    if protocol_success and len(sift_key) > 0:
        print(f"✓ Secure key established by Alice & Bob: {sift_key}")
    else:
        print(f"✗ Protocol FAILED - Key NOT established")
else:
    print("Sift key is empty - no secure key can be established.")


=== EAVESDROPPING DETECTION (REVEALING EVE'S ATTACK) ===
Test positions selected (in sift_key): [24, 22, 18, 12, 9, 5, 4, 3]
Original qubit indices tested: [45, 43, 37, 26, 20, 14, 13, 9]

Alice's original bits: [0, 0, 0, 0, 1, 1, 1, 1]
Bob's received bits:   [1, 0, 0, 0, 0, 1, 1, 1]
Mismatches detected: 2 out of 8 (25.0%)

=== EAVESDROPPER DETECTION RESULT ===
⚠️  EVE DETECTED! ⚠️
Error rate 25.0% exceeds threshold 10.0%
Expected error rate with eavesdropper: ~25% (basis guessing)
ACTION: ABORT PROTOCOL - KEY EXCHANGE FAILED

=== FINAL SECURE KEY STATUS ===
Remaining bits (after removing test bits): [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0]
Final key length: 18 bits
✗ Protocol FAILED - Key NOT established
